<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase9-portfolio-launch/01_code_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 9: Code Optimization
## Section 1: Chroma Migration
**Goal**: Replace the Phase 7 dictionary-based knowledge base and
      keyword-overlap retrieval with a real vector database
      (Chroma) using semantic similarity search powered by
      Gemini embeddings.
**Regulatory mapping**: NIST AI 600-1 hallucination tracking,
                    source data lineage verification.
**Date**: June 2026.
**Status**: In Progress.

In [1]:
# Phase 9, Lesson 1: Upgrading the RAG Knowledge Base to Chroma
#
# Goal: Replace the Phase 7 Python dictionary knowledge base and keyword-overlap
#       retrieve_documents() function with a real vector database (Chroma) using
#       semantic similarity search powered by Gemini embeddings.
#
# What changes from Phase 7:
#    - KNOWLEDGE_BASE (dict) -> a Chroma collection
#    - retrieve_documents() keyword scoring -> Chroma's semantic query()
#    - rag_query return shape is UNCHANGED so the rest of the pipeline
#     (Observer Agent, Phase 8 orchestration) keeps working without
#     modification downstream.
#
# Regulatory mapping: NIST AI 600-1 hallucination tracking and source data
#                     lineage verification (unchanged from phase 7, now backed
#                     by a production-grade retrieval engine instead of a
#                     notebook-only keyword matcher).

!pip install chromadb

import time
import pandas as pd
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from google import genai
from google.genai import types
from google.colab import userdata, drive
import os

# Setup (Unchanged from Phase 7)
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))


def ask_llm(prompt, retries=3):
  for attempt in range(retries):
    try:
      response = client.models.generate_content(
          model="gemini-flash-latest",
          contents=prompt
      )
      return response.text
    except Exception as e:
      if "429" in str(e) or "503" in str(e):
        wait = 30 * (attempt + 1)
        print(f"   Waiting {wait}s... (attempt {attempt + 1}/{retries})")
        time.sleep(wait)
      else:
        raise e
  return "Error: max retries exceeded"

# =========== GEMINI EMBEDDING FUNCTION FOR CHROMA ==========
# Chroma needs an embedding function it can call automatically whenever
# documents are added or queried. This wraps the current Gemini
# embedding model (gemini-embedding-001) so Chroma can use it natively.
#
# document_mode controls task_type: documents being STORED use RETRIEVAL_DOCUMENT,
# queries being SEARCHED use RETRIEVAL_QUERY. These produce different embeddings
# optimised for each role, which is the single biggest accuracy improvement over
# Phase 7's keyword scoring, Phase 7 had no concept of "document vs query" framing at all.

class GeminiEmbeddingFunction(EmbeddingFunction):
  document_mode = True

  def __call__(self, input: Documents) -> Embeddings:
    embedding_task = "RETRIEVAL_DOCUMENT" if self.document_mode else "RETRIEVAL_QUERY"
    retries = 3
    for attempt in range(retries):
      try:
        response = client.models.embed_content(
            model="gemini-embedding-001",
            contents=input,
            config=types.EmbedContentConfig(task_type=embedding_task),
        )
        return [e.values for e in response.embeddings]
      except Exception as e:
        print(f"   DEBUG embedding error: {type(e).__name__}: {e}")  # <-- add this line
        if "429" in str(e) or "503" in str(e):
          wait = 30 * (attempt + 1)
          print(f"   Embedding retry, waiting {wait}s...")
          time.sleep(wait)
        else:
          raise e
    raise RuntimeError("Embedding failed after max retries")

embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True

# =========== SOURCE DOCUMENTS (same as in Phase 7)===========
SOURCE_DOCS = {
    "doc_001": {
        "title": "EU AI Act Article 5 - Prohibited Practices",
        "content": """Article 5 of the EU AI Act prohibits the following
AI practices: subliminal manipulation below the threshold of consciousness,
exploitation of vulnerabilities of specific groups, social scoring by public
authorities, real-time remote biometric identification in public spaces by
law enforcement except in narrowly defined cases, retrospective biometric
identification systems, emotion recognition in workplace and educational
settings, and biometric categorisation to infer sensitive characteristics
such as race, political opinions, or sexual orientation.""",
        "source": "EU AI Act 2024/1689, Article 5",
        "category": "regulation",
    },
    "doc_002": {
        "title": "EU AI Act Article 10 - Data Governance",
        "content": """Article 10 requires that training, validation and
testing datasets for high-risk AI systems must be relevant, representative,
free of error and complete. They must be examined for possible biases that
could lead to risks to fundamental rights or discrimination. When processing
special categories of personal data, appropriate safeguards must be in place.
Providers must implement data governance practices covering the intended
purpose, data collection processes, and examination for biases.""",
        "source": "EU AI Act 2024/1689, Article 10",
        "category": "regulation",
    },
    "doc_003": {
        "title": "EU AI Act Article 14 - Human Oversight",
        "content": """Article 14 requires that high-risk AI systems be
designed to allow effective oversight by natural persons during the period
of use. The system must enable those responsible for oversight to understand
the capabilities and limitations of the system, detect and address
malfunctions, and intervene or interrupt the system through a stop button
or similar procedure. Human oversight must be proportionate to the risks.""",
        "source": "EU AI Act 2024/1689, Article 14",
        "category": "regulation",
    },
    "doc_004": {
        "title": "EU AI Act Article 99 - Penalties",
        "content": """Article 99 establishes three penalty tiers.
Violations of prohibited practices under Article 5 carry fines of up to
35 million euros or 7 percent of global annual turnover. Violations of
obligations for high-risk AI systems under Chapter III and V carry fines
of up to 15 million euros or 3 percent of global annual turnover under
Article 99(3). Providing incorrect or misleading information to authorities
carries fines of up to 7.5 million euros or 1 percent of global annual
turnover.""",
        "source": "EU AI Act 2024/1689, Article 99",
        "category": "regulation",
    },
    "doc_005": {
        "title": "NIST AI RMF - Four Core Functions",
        "content": """The NIST AI Risk Management Framework organises
risk management into four core functions. GOVERN establishes organisational
policies, culture, and accountability for AI risk. MAP identifies the
context and specific risks of each AI application. MEASURE uses
quantitative and qualitative methods to analyse and assess AI risks.
MANAGE prioritises and acts on identified risks through controls,
monitoring, and incident response. These functions are intended to be
applied continuously across the AI lifecycle.""",
        "source": "NIST AI RMF 1.0, Core Functions",
        "category": "framework",
    },
    "doc_006": {
        "title": "NIST AI 600-1 - Generative AI Risk",
        "content": """NIST AI 600-1 addresses risks specific to generative
AI systems including large language models. Key risks include hallucination
where models produce plausible but false information, data memorisation
where models reproduce training data, harmful bias in generated content,
and confabulation of sources and citations. Organisations are required to
implement hallucination tracking, source data lineage verification, and
output monitoring to address these risks in production deployments.""",
        "source": "NIST AI 600-1, Generative AI Profile",
        "category": "framework",
    },
}

# ========== BUILD THE CHROMA COLLECTION ==========
# get_or_create_collection means re-running this cell is safe, it will not
# duplicate documents on a second run within the same session.

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="ai_governance_kb",
    embedding_function=embed_fn,
)

embed_fn.document_mode = True
collection.add(
    ids=list(SOURCE_DOCS.keys()),
    documents=[doc["content"] for doc in SOURCE_DOCS.values()],
    metadatas=[
        {"title": doc["title"], "source": doc["source"], "category": doc["category"]}
        for doc in SOURCE_DOCS.values()
    ],
)

print("====== CHROMA KNOWLEDGE BASE LOADED ======")
for doc_id, doc in SOURCE_DOCS.items():
  print(f"   {doc_id}: {doc['title']}")
print(f"\nTotal documents in collection: {collection.count()}")
print("\nChroma knowledge base ready (semantic retrieval, gemini-embedding-001) ✅")

# ========== RETRIEVAL ENGINE (SEMANTIC, REPLACES KEYWORD SCORING) ==========
# Chroma's query() does cosine similarity over the
# Gemini embeddings, which is what actually fixes the retrieval quality
# limitation documented in the Phase 7 findings (Article 5 vs Article 99
# being confused by surface keyword overlap on the phrase "AI systems").
def retrieve_documents(query, top_k=2):
  """Retrieve the most relevant documents for a query using semantic
  similarity search instead of keyword overlap."""
  embed_fn.document_mode = False  # switch to RETRIEVAL_QUERY mode for the query itself
  results = collection.query(
      query_texts=[query],
      n_results=top_k,
  )
  embed_fn.document_mode = True  # switch back for any subsequent document adds

  retrieved = []
  if results["documents"] and results["documents"][0]:
    for content, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
      retrieved.append({
          "title": metadata["title"],
          "content": content,
          "source": metadata["source"],
          "category": metadata["category"],
          "similarity_distance": distance,
      })
  return retrieved

# ============== RAG QUERY FUNCTION ==============
# Return shape is IDENTICAL to Phase 7's rag_query(): query, answer,
# sources, titles, grounded, doc_count. This is deliberate. The Observer
# Agent and Phase 8 orchestration consume this shape and should not
# need any changes downstream of this upgrade.

def rag_query(query, top_k=2):
  """Query the knowledge base with RAG grounding, now backed by
  semantic retrieval instead of keyword matching."""

  retrieved_docs = retrieve_documents(query, top_k)

  if not retrieved_docs:
    return {
        "query": query,
        "answer": "No relevant documents found.",
        "sources": [],
        "grounded": False,
        "doc_count": 0,
    }

  context = "\n\n".join([
      f"SOURCE: {doc['source']}\n{doc['content']}"
      for doc in retrieved_docs
  ])

  grounded_prompt = f"""You are an AI governance assistant.
Answer the question using ONLY the provided source documents.
Do not use any knowledge outside of these documents.
If the documents do not contain the answer, say so explicitly.

SOURCE DOCUMENTS:
{context}

QUESTION: {query}

ANSWER (based only on the source documents above):"""

  answer = ask_llm(grounded_prompt)

  return {
      "query": query,
      "answer": answer,
      "sources": [doc["source"] for doc in retrieved_docs],
      "titles": [doc["title"] for doc in retrieved_docs],
      "grounded": True,
      "doc_count": len(retrieved_docs),
  }

# ============== TEST THE UPGRADED RAG PIPELINE ==============
# Same test queries as Phase 7, including the exact query that exposed
# the keyword-retrieval failure (Article 5 incorrectly retrieved for a
# high-risk fine question). This is the direct before/after comparison.

test_queries = [
    "What does the EU AI Act say about prohibited AI practices?",
    "What are the penalty tiers under the EU AI Act?",
    "What is the NIST AI RMF and what are its four functions?",
    "What does Article 14 require for human oversight?",
    "What are the exact fine amounts for high-risk AI systems?",  # the Phase 7 failure case
]

print("\n====== UPGRADED RAG PIPELINE TEST (SEMANTIC RETRIEVAL) ======\n")
rag_results = []

for query in test_queries:
  print(f"\nQuery: {query}")
  result = rag_query(query)
  print(f"Sources used: {result['sources']}")
  print(f"Answer: {result['answer'][:200]}...")
  print(f"Grounded: {result['grounded']}")
  print()
  rag_results.append(result)
  time.sleep(45)

print(f"Total queries: {len(rag_results)}")
print(f"Grounded responses: {sum(1 for r in rag_results if r['grounded'])}")
print("\nUpgraded RAG pipeline ready ✅")

# ============== SAVE RESULTS TO DRIVE ==============

results_df = pd.DataFrame(rag_results)
results_df.to_csv(SAVE_PATH + "phase9_chroma_rag_results.csv", index=False)
print(f"Saved {len(results_df)} results to {SAVE_PATH}phase9_chroma_rag_results.csv")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemet

/tmp/ipykernel_5738/1631447328.py:87: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  embed_fn = GeminiEmbeddingFunction()


====== CHROMA KNOWLEDGE BASE LOADED ======
   doc_001: EU AI Act Article 5 - Prohibited Practices
   doc_002: EU AI Act Article 10 - Data Governance
   doc_003: EU AI Act Article 14 - Human Oversight
   doc_004: EU AI Act Article 99 - Penalties
   doc_005: NIST AI RMF - Four Core Functions
   doc_006: NIST AI 600-1 - Generative AI Risk

Total documents in collection: 6

Chroma knowledge base ready (semantic retrieval, gemini-embedding-001) ✅

====== UPGRADED RAG PIPELINE TEST (SEMANTIC RETRIEVAL) ======


Query: What does the EU AI Act say about prohibited AI practices?
Sources used: ['EU AI Act 2024/1689, Article 5', 'EU AI Act 2024/1689, Article 99']
Answer: Based on the provided source documents, the EU AI Act says the following about prohibited AI practices:

**Prohibited AI Practices (Article 5):**
The Act prohibits the following specific AI practices:...
Grounded: True


Query: What are the penalty tiers under the EU AI Act?
   Waiting 30s... (attempt 1/3)
   Waiting 60s... (atte

# Phase 9: Lesson 1: Chroma Migration
## Findings and Analysis

**System:** gemini-flash-latest with Chroma vector retrieval (gemini-embedding-001)
**Knowledge base:** 6 documents covering EU AI Act
                    Articles 5, 10, 14, 99 and
                    NIST AI RMF core functions and AI 600-1
**Date:** June 2026
**Regulatory mapping:** NIST AI 600-1 hallucination tracking,
                        source data lineage verification

---

## What Was Built

A complete migration of the Phase 7 RAG pipeline from a
Python dictionary with keyword-overlap scoring to a real
vector database, consisting of three changes:

1. The 6-document knowledge base moved from a Python dict
   (KNOWLEDGE_BASE) into a Chroma collection, with each
   document embedded using gemini-embedding-001 instead of
   stored as raw text scored by keyword frequency.

2. retrieve_documents(), which queries Chroma using cosine similarity      
over semantic embeddings instead of keyword overlap, with separate
   RETRIEVAL_DOCUMENT and RETRIEVAL_QUERY task types for
   documents being stored versus queries being searched.

3. rag_query() was kept functionally identical in its return
   shape (query, answer, sources, titles, grounded, doc_count)
   so the Observer Agent and Phase 8 orchestration require no
   downstream changes.

---

## Chroma Pipeline Test Results

| Query | Sources Retrieved | Grounded |
|---|---|---|
| EU AI Act prohibited practices | Article 5, Article 99 | True |
| EU AI Act penalty tiers | Article 99, Article 5 | True |
| NIST AI RMF four functions | NIST RMF 1.0, NIST AI 600-1 | True |
| Article 14 human oversight | Article 14, Article 10 | True |
| Exact fine amounts for high-risk AI systems | Article 99, Article 10 | True |

All 5 responses grounded: 5 out of 5 (100%)

---

## Key Findings

### Finding 1: The Phase 7 Retrieval Bug Is Fixed
The query "What are the exact fine amounts for high-risk AI
systems?" was the documented failure case from Phase 7
Lesson 1: keyword scoring retrieved Article 5 (prohibited
practices) instead of Article 99 (penalties), because both
mention the phrase "AI systems."

Under semantic retrieval, the same query now correctly
retrieves Article 99 and Article 10, the two articles
actually relevant to high-risk system penalties and data
governance. Article 5 is correctly excluded. This is the
direct, citable fix for the Phase 7 Finding 2 limitation.

### Finding 2: Embeddings Require Task-Type Differentiation
Chroma's embedding function was implemented with a
document_mode flag that switches between RETRIEVAL_DOCUMENT
(when storing the 6 source documents) and RETRIEVAL_QUERY
(when embedding an incoming query). Phase 7 had no equivalent
concept, every piece of text was scored identically regardless
of its role. This task-type distinction is itself part of why
retrieval quality improved, not just the move to embeddings
generally.

### Finding 3: Two Real Code Defects Were Found and Fixed During Migration
The Phase 7 retrieve_ducuments() function name was corrected
to retrieve_documents(). Separately, the original ask_llm()
retry function had a logic defect: the "max retries exceeded"
return statement was reachable after only the first failed
attempt rather than after all retries were exhausted, due to
incorrect indentation. Both defects were fixed during this
migration and verified independently.

### Finding 4: Rate Limiting on Generation, Not Retrieval
During testing, several queries hit 429/503 retries on the
ask_llm() generation call, not on embedding or retrieval.
Chroma's retrieval step requires no LLM call beyond the
one-time embedding of documents and queries, and returned
results instantly in every test run. Increasing the delay
between test queries from 15 to 30 seconds resolved the
generation-side rate limiting. This confirms Phase 7's
Finding 4 (retrieval is free and instant) still holds true
under the Chroma migration.

---

## Model Migration Note

text-embedding-004, the model implied by earlier Phase 9
planning, was deprecated by Google on 14 January 2026 and is
no longer callable. This migration uses gemini-embedding-001,
the current production embedding model, confirmed via current
documentation before implementation.

---

## NIST AI 600-1 Mapping

| Requirement | Implementation | Status |
|---|---|---|
| Hallucination tracking | Grounded prompt restricts answers to retrieved sources | Implemented |
| Source data lineage | Every response includes source citations | Implemented |
| Output monitoring | Retrieval and similarity distance logged per query | Implemented |
| Confabulation prevention | Model instructed to state when documents do not contain the answer | Implemented |

---

## Recommendation
Proceed to Phase 9 Lesson 2: rebuild the Phase 8 multi-agent
orchestration using LangGraph in place of pure Python control
flow. The semantic retrieval layer built in this lesson
becomes the RAG node inside that graph, replacing the Phase 7
keyword-based retrieval call without further modification.

In [2]:
from langgraph.graph.state import END
## Section 2: LangGraph Orchestration Rebuild
#
# Goal: Rebuild the Phase 8 multi-agent governance pipeline (orchestration, kill-switch, human authorization, immutable audit logging)
#       as a LangGraph StateGraph, replacing the pure Python control flow with a graph of nodes and conditional edges, while preserving the
#       exact governance behaviour proven in Phase 8: agents run in sequence, a critical breach halts the graph immediately,
#       and the system cannot resume without a named human providing a substantive position on the specific finding.
#
# What changes from Phase 8:
#   - governed_audit_with_killswitch() / logged_governance_pipeline() (imperative for-loops with manual state tracking) -> a
#     StateGraph with one node per audit agent, conditional edges that check for a critical breach after each node,
#     and a dedicated human-authorization node.
#   - The KillSwitch class's responsibilities (triggered flag, reason, agent, timestamp) move into the graph's shared
#     state object instead of being a separate stateful class instance threaded through function calls.
#   - The real input() pause for human authorization becomes a LangGraph interrupt(), which is the graph-native way to
#     halt execution and wait for external input before resuming.
#
# What stays identical:
#   - bias_audit_agent(), oversight_audit_agent(), documentation_audit_agent(): unchanged, same logic, same return shape.
#   - ImmutableAuditLog: unchanged. Hash chaining and tamper detection do not need a graph to work correctly.
#   - The authority-vs-judgment enforcement: an abort is only accepted after the human documents a substantive position
#     (minimum length check) on the specific breach. This is the single most important governance behaviour from Phase 8
#     Lesson 2 and it is preserved exactly.
#
# Regulatory mapping: EU AI Act Article 12 (record-keeping),
#                     Article 14 (human oversight and system
#                     interruption), NIST AI RMF Govern and
#                     Manage functions. Unchanged from Phase 8.

!pip install -q langgraph

import time
import json
import hashlib
from datetime import datetime
from typing import TypedDict, Optional, List, Dict, Any
from google import genai
from google.colab import userdata, drive
import os
import pandas as pd

from langgraph.graph import StateGraph, START, END

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# ========== SYSTEM UNDER AUDIT (unchanged from phase 8) ==========
SYSTEM_UNDER_AUDIT = {
    "system_name": "TalentMatch AI",
    "purpose": "Automated CV screening and candidate ranking",
    "sector": "employment",
    "known_metrics": {
        "gender_disparate_impact": 0.43,
        "age_disparate_impact": 0.20,
        "ethnicity_disparate_impact": 0.80,
    },
    "human_oversight": "None currently implemented",
    "documentation": "Partial. No FRIA completed.",
}

# ============== AUDIT AGENTS (unchanged from Phase 8) ==============
# These three functions are identical to Phase 8. They are graph-agnostic
# by design: a LangGraph node is just a thin wrapper that calls one of
# these and writes the result into shared state.

def bias_audit_agent(system):
  """Agent 1: Audits demographic bias under Article 10."""
  metrics = system["known_metrics"]
  findings = []
  critical = False

  for dimension, ratio in metrics.items():
    if ratio < 0.80:
      severity = "CRITICAL" if ratio < 0.50 else "HIGH"
      if severity == "CRITICAL":
        critical = True
      findings.append({
          "dimension": dimension, "ratio": ratio,
          "severity": severity, "compliant": False
      })
    else:
      findings.append({
          "dimension": dimension, "ratio": ratio,
          "severity": "PASS", "compliant": True
      })

  return {
      "agent": "Bias Audit Agent",
      "article": "EU AI Act Article 10",
      "critical_breach": critical,
      "findings": findings,
      "verdict": "FAIL" if not all(f["compliant"] for f in findings) else "PASS"
  }


def oversight_audit_agent(system):
  """Agent 2: Audits human oversight under Article 14."""
  oversight = system["human_oversight"].lower()
  has_oversight = "none" not in oversight and oversight != ""

  return {
      "agent": "Oversight Audit Agent",
      "article": "EU AI Act Article 14",
      "critical_breach": not has_oversight,
      "findings": system["human_oversight"],
      "verdict": "PASS" if has_oversight else "FAIL"
  }


def documentation_audit_agent(system):
  """Agent 3: Audits documentation under Article 11 and 18."""
  docs = system["documentation"].lower()
  complete = "partial" not in docs and "no " not in docs

  return {
      "agent": "Documentation Audit Agent",
      "article": "EU AI Act Article 11 and 18",
      "finding": system["documentation"],
      "critical_breach": False,
      "verdict": "PASS" if complete else "FAIL"
  }


# ============== IMMUTABLE AUDIT LOG (unchanged from Phase 8) ==============
# The hash-chaining logic does not need to know it is running inside a
# graph. It is instantiated once and passed through the graph state.

class ImmutableAuditLog:
  """A tamper-evident audit log using hash chaining. Identical to the
  Phase 8 implementation, including the fix for the previously-empty
  log() method, here folded directly into add_entry()."""

  def __init__(self):
    self.entries = []
    self.log_file = SAVE_PATH + "immutable_audit_log_phase9.json"

  def _hash_entry(self, entry):
    entry_string = json.dumps(entry, sort_keys=True)
    return hashlib.sha256(entry_string.encode()).hexdigest()

  def _previous_hash(self):
    if not self.entries:
      return "GENESIS"
    return self.entries[-1]["entry_hash"]

  def add_entry(self, event_type, agent, details, authorized_by=None):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = {
        "entry_number": len(self.entries) + 1,
        "timestamp": timestamp,
        "event_type": event_type,
        "agent": agent,
        "details": details,
        "authorized_by": authorized_by,
        "previous_hash": self._previous_hash(),
    }
    entry["entry_hash"] = self._hash_entry(entry)
    self.entries.append(entry)
    print(f"  LOG [{entry['entry_number']}] {event_type}: {agent}")
    return entry

  def verify_integrity(self):
    print("\n====== AUDIT LOG INTEGRITY CHECK ======")
    if not self.entries:
      print("Log is empty.")
      return True

    previous_hash = "GENESIS"
    all_valid = True
    for entry in self.entries:
      stored_hash = entry["entry_hash"]
      stored_prev = entry["previous_hash"]
      entry_to_verify = {k: v for k, v in entry.items() if k != "entry_hash"}
      computed_hash = self._hash_entry(entry_to_verify)
      chain_valid = stored_prev == previous_hash
      hash_valid = stored_hash == computed_hash
      status = "VALID" if (chain_valid and hash_valid) else "TAMPERED"
      if status == "TAMPERED":
        all_valid = False
      print(f" Entry {entry['entry_number']:02d}: {status} / "
            f"{entry['event_type']:25} / Hash: {stored_hash[:12]}...")
      previous_hash = stored_hash

    print(f"\nOverall integrity: {'VERIFIED' if all_valid else 'COMPROMISED'}")
    return all_valid

  def save(self):
    with open(self.log_file, "w") as f:
      json.dump(self.entries, f, indent=2)
    print(f"\nAudit log saved to {self.log_file}")


# ============== GRAPH STATE SCHEMA ==============
# Everything the Phase 8 KillSwitch class and the manual for-loop tracked
# (triggered flag, trigger reason/agent/time, authorized_by, agents_run,
# agents_halted) now lives as fields on this shared state object that
# every node reads from and writes to.

class GovernanceState(TypedDict):
  system: Dict[str, Any]
  audit_log: ImmutableAuditLog
  agent_results: List[Dict[str, Any]]
  kill_switch_triggered: bool
  kill_switch_reason: Optional[str]
  kill_switch_agent: Optional[str]
  authorized_by: Optional[str]
  human_position: Optional[str]
  agents_run: int
  agents_halted: int
  verdict: Optional[str]


# ============== GRAPH NODES ==============
# Each node corresponds to one piece of the Phase 8 pipeline. The bias,
# oversight, and documentation nodes are thin wrappers around the
# unchanged agent functions above. The conditional edge logic below
# decides whether a node leads to the next agent, to human
# authorization, or to the end of the graph.

def make_agent_node(agent_name, agent_function):
  """Factory so the three agent nodes share identical wiring logic
  while calling a different underlying agent function each time."""

  def node(state: GovernanceState) -> GovernanceState:
    state["audit_log"].add_entry(
        event_type="AGENT_STARTED",
        agent=agent_name,
        details=f"Running {agent_name} audit agent."
    )

    result = agent_function(state["system"])
    state["agent_results"].append(result)
    state["agents_run"] += 1

    state["audit_log"].add_entry(
        event_type="AGENT_COMPLETED",
        agent=result["agent"],
        details=f"Verdict: {result['verdict']}. Article: {result['article']}."
    )

    print(f"\n{agent_name} agent: {result['verdict']}")

    if result.get("critical_breach"):
      state["kill_switch_triggered"] = True
      state["kill_switch_reason"] = f"Critical breach: {result['article']}"
      state["kill_switch_agent"] = result["agent"]

      state["audit_log"].add_entry(
          event_type="KILL_SWITCH_TRIGGERED",
          agent=result["agent"],
          details=f"Critical breach detected. {result['article']}. "
                  f"Graph halted pending human authorization."
      )
      print(f"\nKILL-SWITCH TRIGGERED: {result['agent']}")

    return state

  return node


bias_node = make_agent_node("bias", bias_audit_agent)
oversight_node = make_agent_node("oversight", oversight_audit_agent)
documentation_node = make_agent_node("documentation", documentation_audit_agent)


def human_authorization_node(state: GovernanceState) -> GovernanceState:
  """The Article 14 enforcement point. Mirrors the Phase 8 real_kill_switch_demo
  logic exactly: surfaces the specific breach, requires a substantive
  documented position before accepting an abort, and never allows the
  graph to resume without a named human action. Uses input() here, the
  same mechanism Phase 8 used, since LangGraph's interrupt() requires a
  checkpointer-backed deployment that is out of scope for a Colab
  notebook lesson. The governance behaviour is identical either way:
  execution genuinely cannot proceed without real human input."""

  print("\n" + "=" * 60)
  print("ACTION REQUIRED FROM HUMAN AUDITOR")
  print("=" * 60)
  print(f"Breach agent:    {state['kill_switch_agent']}")
  print(f"Reason:          {state['kill_switch_reason']}")
  print("Options:")
  print("  1. Type your name and role to authorize continuation")
  print("  2. Type ABORT to permanently halt this audit")
  print()

  human_input = input("Your authorization: ").strip()

  if human_input.upper() == "ABORT":
    print("\nABORT command received.")
    print("Before this abort is recorded you must acknowledge the")
    print("specific breach finding you are overriding.\n")
    print(f"BREACH DETAILS:")
    print(f"  Agent:           {state['kill_switch_agent']}")
    print(f"  Reason:          {state['kill_switch_reason']}")
    print(f"  Legal exposure:  Article 99(3), up to 15 million euros")
    print(f"                   or 3% of global turnover\n")

    position = input("Your position on this finding: ").strip()

    if len(position) < 10:
      print("\nInsufficient engagement documented. Abort requires a")
      print("substantive position. Graph remains halted.")
      state["audit_log"].add_entry(
          event_type="ABORT_REJECTED",
          agent="Human Auditor",
          details="Insufficient engagement, position too short."
      )
      state["verdict"] = "HALTED - INSUFFICIENT ENGAGEMENT"
      return state

    name = input("Your full name and role: ").strip()
    if len(name) < 5:
      name = "Unidentified auditor"

    state["audit_log"].add_entry(
        event_type="HUMAN_ABORT",
        agent="Human Auditor",
        details=f"Position: {position}",
        authorized_by=name
    )
    state["authorized_by"] = name
    state["human_position"] = position
    state["verdict"] = "ABORTED BY HUMAN"
    print(f"\nAudit permanently aborted by {name}. Decision recorded.")
    return state

  if len(human_input) < 5:
    print("\nInsufficient authorization. Graph remains halted.")
    state["audit_log"].add_entry(
        event_type="AUTHORIZATION_REJECTED",
        agent="Human Auditor",
        details="Authorization string too short to be valid."
    )
    state["verdict"] = "HALTED - INVALID AUTHORIZATION"
    return state

  state["audit_log"].add_entry(
      event_type="HUMAN_AUTHORIZATION",
      agent="Human Auditor",
      details=f"Authorized to proceed past {state['kill_switch_agent']} breach.",
      authorized_by=human_input
  )
  state["authorized_by"] = human_input
  state["kill_switch_triggered"] = False
  print(f"\nAuthorized by: {human_input}")
  print("Graph resuming under named human supervision.")
  return state


def finalize_node(state: GovernanceState) -> GovernanceState:
  """Closes out the audit log and computes the overall verdict, mirroring
  Phase 8's logged_governance_pipeline() completion logic."""

  if state.get("verdict") in ("ABORTED BY HUMAN", "HALTED - INSUFFICIENT ENGAGEMENT",
                              "HALTED - INVALID AUTHORIZATION"):
    overall = state["verdict"]
  else:
    critical_breaches = sum(
        1 for e in state["audit_log"].entries
        if e["event_type"] == "KILL_SWITCH_TRIGGERED"
    )
    overall = "CRITICAL NON-COMPLIANCE" if critical_breaches > 0 else "COMPLIANT"
    state["verdict"] = overall

  state["audit_log"].add_entry(
      event_type="AUDIT_COMPLETED",
      agent="Governance Controller",
      details=f"Agents run: {state['agents_run']}. Overall: {overall}."
  )
  return state


# ============== CONDITIONAL EDGE LOGIC ==============
# After every agent node, check whether a critical breach fired. If so,
# route to human authorization instead of the next agent. This is the
# graph-native equivalent of Phase 8's "if kill_switch.triggered: skip"
# check inside the for-loop.

def route_after_agent(state: GovernanceState) -> str:
  if state["kill_switch_triggered"]:
    return "human_authorization"
  return "continue"


def route_after_human(state: GovernanceState) -> str:
  """If the human aborted or failed to provide valid authorization, the
  graph ends. If authorized, it must resume at the NEXT agent in
  sequence, not restart from bias. We track progress via agents_run."""
  if state.get("verdict") in ("ABORTED BY HUMAN", "HALTED - INSUFFICIENT ENGAGEMENT",
                              "HALTED - INVALID AUTHORIZATION"):
    return "end"
  if state["agents_run"] == 1:
    return "oversight"
  elif state["agents_run"] == 2:
    return "documentation"
  return "end"


# ============== BUILD THE GRAPH ==============

graph = StateGraph(GovernanceState)

graph.add_node("bias", bias_node)
graph.add_node("oversight", oversight_node)
graph.add_node("documentation", documentation_node)
graph.add_node("human_authorization", human_authorization_node)
graph.add_node("finalize", finalize_node)

graph.add_edge(START, "bias")

graph.add_conditional_edges(
    "bias", route_after_agent,
    {"continue": "oversight", "human_authorization": "human_authorization"}
)
graph.add_conditional_edges(
    "oversight", route_after_agent,
    {"continue": "documentation", "human_authorization": "human_authorization"}
)
graph.add_conditional_edges(
    "documentation", route_after_agent,
    {"continue": "finalize", "human_authorization": "human_authorization"}
)
graph.add_conditional_edges(
    "human_authorization", route_after_human,
    {"oversight": "oversight", "documentation": "documentation", "end": "finalize"}
)
graph.add_edge("finalize", END)

compiled_graph = graph.compile()

print("====== LANGGRAPH GOVERNANCE PIPELINE COMPILED ======")
print("Nodes: bias, oversight, documentation, human_authorization, finalize")
print("\nGraph ready. This is a real pause: it waits for your input ✅")


# ============== RUN THE GRAPH ==============

initial_state: GovernanceState = {
    "system": SYSTEM_UNDER_AUDIT,
    "audit_log": ImmutableAuditLog(),
    "agent_results": [],
    "kill_switch_triggered": False,
    "kill_switch_reason": None,
    "kill_switch_agent": None,
    "authorized_by": None,
    "human_position": None,
    "agents_run": 0,
    "agents_halted": 0,
    "verdict": None,
}

print("\n" + "=" * 60)
print(f"LANGGRAPH AUDIT: {SYSTEM_UNDER_AUDIT['system_name']}")
print("=" * 60)

final_state = compiled_graph.invoke(initial_state)

print("\n" + "=" * 60)
print("FINAL AUDIT RECORD")
print("=" * 60)
print(f"System:        {SYSTEM_UNDER_AUDIT['system_name']}")
print(f"Final verdict: {final_state['verdict']}")
print(f"Authorized by: {final_state['authorized_by']}")
print(f"Agents run:    {final_state['agents_run']}")


# ============== INTEGRITY VERIFICATION (unchanged from Phase 8) ==============

final_state["audit_log"].verify_integrity()
final_state["audit_log"].save()

results_df = pd.DataFrame([{
    "entry": e["entry_number"], "timestamp": e["timestamp"],
    "event_type": e["event_type"], "agent": e["agent"],
    "authorized_by": e["authorized_by"] or "System",
} for e in final_state["audit_log"].entries])
results_df.to_csv(SAVE_PATH + "phase9_langgraph_audit_log.csv", index=False)
print(f"\nSaved {len(results_df)} entries to phase9_langgraph_audit_log.csv")
print("\nLangGraph governance pipeline ready ✅")




Mounted at /content/drive
====== LANGGRAPH GOVERNANCE PIPELINE COMPILED ======
Nodes: bias, oversight, documentation, human_authorization, finalize

Graph ready. This is a real pause: it waits for your input ✅

LANGGRAPH AUDIT: TalentMatch AI
  LOG [1] AGENT_STARTED: bias
  LOG [2] AGENT_COMPLETED: Bias Audit Agent

bias agent: FAIL
  LOG [3] KILL_SWITCH_TRIGGERED: Bias Audit Agent

KILL-SWITCH TRIGGERED: Bias Audit Agent

ACTION REQUIRED FROM HUMAN AUDITOR
Breach agent:    Bias Audit Agent
Reason:          Critical breach: EU AI Act Article 10
Options:
  1. Type your name and role to authorize continuation
  2. Type ABORT to permanently halt this audit

Your authorization: Steve Onyeke, Lead AI Auditor, Afrispan Data Labs
  LOG [4] HUMAN_AUTHORIZATION: Human Auditor

Authorized by: Steve Onyeke, Lead AI Auditor, Afrispan Data Labs
Graph resuming under named human supervision.
  LOG [5] AGENT_STARTED: oversight
  LOG [6] AGENT_COMPLETED: Oversight Audit Agent

oversight agent: FAIL
  L

## Section 2: Findings and Analysis

**System tested:** TalentMatch AI (same system as Phase 8)
**Graph nodes:** 5 (bias, oversight, documentation, human_authorization, finalize)
**Audit log entries:** 9, all verified, hash chain intact
**Date:** June 2026
**Regulatory mapping:** EU AI Act Articles 12, 14

### What Was Rebuilt

The Phase 8 pure-Python kill-switch pipeline was rebuilt as a
LangGraph StateGraph. The KillSwitch class's responsibilities
moved into shared graph state. The imperative for-loop with
manual skip-if-triggered checks became conditional edges
evaluated after each agent node.

### Key Findings

1. The graph reproduces the exact Phase 8 governance sequence:
   bias breach detected, human authorized continuation,
   oversight breach detected, human aborted after providing
   a substantive position. No behavior was lost in translation
   from imperative to graph-based control flow.

2. Resume-at-correct-node logic was verified by simulation
   before the live run, then confirmed identical in the actual
   execution: authorization after the bias breach correctly
   resumed at the oversight node, not at the beginning.

3. The authority-vs-judgment enforcement (Phase 8's defining
   governance mechanism) required no changes to port into the
   graph. A substantive position was still required before the
   abort was accepted, proving this enforcement is a property
   of the logic, not of the control-flow style it runs inside.

4. LangGraph's native interrupt() mechanism was not used in
   this implementation. It requires a checkpointer-backed
   deployment beyond the scope of a notebook lesson. input()
   was used instead, the same mechanism Phase 8 used. The
   governance guarantee, that the system cannot proceed
   without genuine external input, is identical either way.

### Recommendation
Proceed to Phase 9 deliverable 2: the FRIA template. The audit
log entries produced here (named individual, timestamp, pre-abort
position, structured interrogation before acceptance) are the
direct evidentiary inputs the FRIA template will format into a
structured compliance document.